## This notebook is used to: 
### i) transform proper motions to Galactic coordinates
### ii) correct for Galactic rotation 
### iii) convert back to equatorial motions with respect to the surrounding medium.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.coordinates as coord
import astropy.units as u
from uncertainties import ufloat
from uncertainties.umath import *
import ast

AttributeError: module 'numpy' has no attribute 'asscalar'

In [ ]:
# reading the data from the file
with open('BD+43°3654.txt') as f:
    data = f.read()
      
# reconstructing the data as a dictionary
data = ast.literal_eval(data)

print(data)

### First we define some general functions

In [ ]:
def galactic_velocity(dist, b, l, A, B, C, K, U, V, W):
    '''Galactic velocity curve from Wang et al. 2021 (https://arxiv.org/pdf/2103.10232.pdf, Eqs.7-9). 
    Arguments:
    dist: distance in kpc
    b, l: galactic coordinates in radians
    A, B, C, K: Oort constants in km/s/kpc
    U, V, W: Solar velocity in km/s'''

    #cb = np.cos(b)
    #sb = np.sin(b)
    #cl = np.cos(l)
    #sl = np.sin(l)
    #c2l = np.cos(2*l)
    #s2l = np.sin(2*l)
    #d_adim = dist/u.kpc
    cb = cos(b)
    sb = sin(b)
    cl = cos(l)
    sl = sin(l)
    c2l = cos(2*l)
    s2l = sin(2*l)
    d_adim = dist
    
    V_r0 = -U*cl*cb - V*sl*cb - W*sb + d_adim * (K + A*s2l + C*c2l) * cb*cb 
                    
    V_l0 = U*sl - V*cl + d_adim * (A*c2l + B - C*s2l) * cb
    
    V_b0 = U*cl*sb + V*sl*sb - W*cb - d_adim * (K + A*s2l + C*c2l) * sb*cb  
    
    return V_r0, V_l0, V_b0


def velocity_to_propermotion(dist, V_l0, V_b0):
    '''Transform V_l0 and V_b0 to mu_l0 and mu_b0.
    dist: distance in kpc
    V_l0, V_b0: velocities in km/s'''
    
    #mu_l0 = 0.211/(dist/u.kpc) * V_l0 * u.mas/u.yr
    #mu_b0 = 0.211/(dist/u.kpc) * V_b0 * u.mas/u.yr
    mu_l0 = 0.211/dist * V_l0
    mu_b0 = 0.211/dist * V_b0 
    
    return mu_l0, mu_b0


def oort_constants(model):
    '''Retrieve the Oort constants for one of different models:
    C07: Comerón & Pascquali 2007
    B19: Bobylev & Bajkova 2019 (OSCs)
    W21: Wang et al. 2021 (https://arxiv.org/pdf/2103.10232.pdf)'''
    
    if model == 'C07':
        A = 12.5
        B = -12.5
        C = 0.0
        K = 0.0
        U = 7.0
        V = 14.0
        W = 7.0
    elif model == 'B19':
        A = 16.40
        B = -12.31
        C = 0.0
        K = 0.0
        U = 8.53
        V = 11.22
        W = 7.83
    elif model == 'W21':
        A = 16.31
        B = -11.99
        C = -3.10
        K = -1.25
        U = 11.69
        V = 10.16
        W = 7.67
    else:
        print('ERROR! wrong model')
 
    return A, B, C, K, U, V, W

### Second we introduce the barycentric coordinates and velocity in the ICRS for the object

In [ ]:
# Gaia DR3 data
BD = coord.SkyCoord(ra=data['ra']*u.degree, dec=data['dec']*u.degree,
                    distance=(data['parallax']*u.mas).to(u.kpc, u.parallax()),
                    pm_ra_cosdec=data['mu_ra_cosdec']*u.mas/u.yr,
                    pm_dec=data['mu_dec']*u.mas/u.yr,
                    radial_velocity=data['V_r']*u.km/u.s, 
                    frame='icrs')

parallax = ufloat(data['parallax'], data['parallax_err'])
ra_err = radians(data['ra_err'])
dec_err = radians(data['dec_err'])
pm_ra_err = data['mu_ra_cosdec_err']
pm_dec_err = data['mu_dec_err']
v_r_err = data['V_r_err']
# Print the values to check they are OK
#print(BD.galactic)
# Save in shorter-name variables and convert to ufloatable objects
alpha = ufloat(BD.ra.radian,ra_err)
delta = ufloat(BD.dec.radian,dec_err)
l, b, D = BD.galactic.l, BD.galactic.b, BD.galactic.distance
mu_l, mu_b = BD.galactic.pm_l_cosb, BD.galactic.pm_b
V_r = ufloat(BD.galactic.radial_velocity.value, v_r_err)
print(f'{D =}')
print(f'{l =}, {b=}')
print(f'{mu_l =}, {mu_b=}')
print(f'{V_r =}')

In [ ]:
Ag = np.array([[-0.0548755604162154,-0.8734370902348850,-0.4838350155487132],\
              [0.4941094278755837,-0.4448296299600112,0.7469822444972189],\
              [-0.8676661490190047,-0.1980763734312015,0.4559837761750669]])

Agt = Ag.transpose()

dist = 1./parallax

l_gal = atan2(Ag[1,0]*cos(alpha)*cos(delta)+Ag[1,1]*sin(alpha)*cos(delta)+Ag[1,2]*sin(delta),\
             Ag[0,0]*cos(alpha)*cos(delta)+Ag[0,1]*sin(alpha)*cos(delta)+Ag[0,2]*sin(delta))

b_gal = atan2(Ag[2,0]*cos(alpha)*cos(delta)+Ag[2,1]*sin(alpha)*cos(delta)+Ag[2,2]*sin(delta),\
             sqrt((Ag[1,0]*cos(alpha)*cos(delta)+Ag[1,1]*sin(alpha)*cos(delta)+Ag[1,2]*sin(delta))**2+\
             (Ag[0,0]*cos(alpha)*cos(delta)+Ag[0,1]*sin(alpha)*cos(delta)+Ag[0,2]*sin(delta))**2))

mul = -sin(l_gal)*( (-Ag[0,0]*sin(alpha)+Ag[0,1]*cos(alpha))*BD.pm_ra_cosdec\
    -(Ag[0,0]*cos(alpha)*cos(delta)+Ag[0,1]*sin(alpha)*sin(delta)-Ag[0,2]*cos(delta))*BD.pm_dec)\
+cos(l_gal)*( (-Ag[1,0]*sin(alpha)+Ag[1,1]*cos(alpha))*BD.pm_ra_cosdec\
    -(Ag[1,0]*cos(alpha)*cos(delta)+Ag[1,1]*sin(alpha)*sin(delta)-Ag[1,2]*cos(delta))*BD.pm_dec)

mub = -cos(l_gal)*sin(b_gal)*( (-Ag[0,0]*sin(alpha)+Ag[0,1]*cos(alpha))*BD.pm_ra_cosdec\
    -(Ag[0,0]*cos(alpha)*cos(delta)+Ag[0,1]*sin(alpha)*sin(delta)-Ag[0,2]*cos(delta))*BD.pm_dec)\
-sin(l_gal)*sin(b_gal)*( (-Ag[1,0]*sin(alpha)+Ag[1,1]*cos(alpha))*BD.pm_ra_cosdec\
    -(Ag[1,0]*cos(alpha)*cos(delta)+Ag[1,1]*sin(alpha)*sin(delta)-Ag[1,2]*cos(delta))*BD.pm_dec)\
+cos(b_gal)*( (-Ag[2,0]*sin(alpha)+Ag[2,1]*cos(alpha))*BD.pm_ra_cosdec\
    -(Ag[2,0]*cos(alpha)*cos(delta)+Ag[2,1]*sin(alpha)*sin(delta)-Ag[2,2]*cos(delta))*BD.pm_dec)

mu_alpha_corr = -sin(alpha)*( (-Agt[0,0]*sin(l_gal)+Agt[0,1]*cos(l_gal))*mul\
    -(Agt[0,0]*cos(l_gal)*cos(b_gal)+Agt[0,1]*sin(l_gal)*sin(b_gal)-Agt[0,2]*cos(b_gal))*mub)\
+cos(alpha)*( (-Agt[1,0]*sin(l_gal)+Agt[1,1]*cos(l_gal))*mul\
    -(Agt[1,0]*cos(l_gal)*cos(b_gal)+Agt[1,1]*sin(l_gal)*sin(b_gal)-Agt[1,2]*cos(b_gal))*mub)

print('distance =', dist, 'kpc' )


### Third, we calculate local Galactic velocity field and the object proper motion w.r.t. local medium

In [ ]:
#A, B, C, K, U, V, W = oort_constants('C07')
A, B, C, K, U, V, W = oort_constants('B19')
#A, B, C, K, U, V, W = oort_constants('W21')


V_r0, V_l0, V_b0 = galactic_velocity(dist, b_gal, l_gal, A, B, C, K, U, V, W)

mu_l0, mu_b0 = velocity_to_propermotion(dist, V_l0, V_b0)

print(' Galactic rotation in mas/yr:')
print('(\u03BC_l)0=' + str(mu_l0) + ', (\u03BC_b)0=' + str(mu_b0) )

# Proper motions w.r.t. the surrounding medium:
mu_l_corr = mul.value - mu_l0
mu_b_corr = mub.value - mu_b0

print('\n Corrected values:')
print('\u03BC_l=' + str(mu_l_corr) + ', \u03BC_b=' + str(mu_b_corr) )


#Calculate mu_alpha, mu_dec, and V_tan

mu_alpha_corr = -sin(alpha)*( (-Agt[0,0]*sin(l_gal)+Agt[0,1]*cos(l_gal))*mu_l_corr\
    -(Agt[0,0]*cos(l_gal)*cos(b_gal)+Agt[0,1]*sin(l_gal)*sin(b_gal)-Agt[0,2]*cos(b_gal))*mu_b_corr)\
+cos(alpha)*( (-Agt[1,0]*sin(l_gal)+Agt[1,1]*cos(l_gal))*mu_l_corr\
    -(Agt[1,0]*cos(l_gal)*cos(b_gal)+Agt[1,1]*sin(l_gal)*sin(b_gal)-Agt[1,2]*cos(b_gal))*mu_b_corr)

mu_dec_corr = -cos(alpha)*sin(delta)*( (-Agt[0,0]*sin(l_gal)+Agt[0,1]*cos(l_gal))*mu_l_corr\
    -(Agt[0,0]*cos(l_gal)*cos(b_gal)+Agt[0,1]*sin(l_gal)*sin(b_gal)-Agt[0,2]*cos(b_gal))*mu_b_corr)\
-sin(alpha)*sin(delta)*( (-Agt[1,0]*sin(l_gal)+Agt[1,1]*cos(l_gal))*mu_l_corr\
    -(Agt[1,0]*cos(l_gal)*cos(b_gal)+Agt[1,1]*sin(l_gal)*sin(b_gal)-Agt[1,2]*cos(b_gal))*mu_b_corr)\
+cos(delta)*( (-Agt[2,0]*sin(l_gal)+Agt[2,1]*cos(l_gal))*mu_l_corr\
    -(Agt[2,0]*cos(l_gal)*cos(b_gal)+Agt[2,1]*sin(l_gal)*sin(b_gal)-Agt[2,2]*cos(b_gal))*mu_b_corr)

#angle = degrees(atan(mub.value/(-mul.value)))
angle = degrees(atan(BD.pm_dec/BD.pm_ra_cosdec))
#angle = np.arctan(icrs.pm_ra_cosdec/icrs.pm_dec).to(u.deg)
angle_corr = degrees(atan(mu_l_corr/(mu_b_corr)))
print( '\n angle [deg] =', str(angle) )
print( '\n angle_corr [deg] =', str(angle_corr) )

# Vl= 4.74*r*μ_l_cosb
# Vb= 4.74*r*μ_b
# V_t = 4.74*D*sqrt(μ_l_cosb^2 + μ_b^2)
V_t = 4.74*dist * sqrt(mu_l_corr**2 + mu_b_corr**2)
print( '\n V_t [km/s]=', str(V_t) )

V_r_corr = V_r - V_r0
print( '\n V_r [km/s]=', str(V_r_corr) ) 